# Multitask D-Optimal Design Preprocessing

This notebook prepares the Ames Housing dataset for the multitask D-optimal design experiments. It cleans the selected numeric columns, clusters the observations into community-level design points, builds the task-specific design matrices, and saves all artifacts consumed by `04 Multitask D_optimal Design.ipynb`.


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

SEED = 7
rng = np.random.default_rng(SEED)

data_csv = Path("AmesHousing.csv")
preprocessed_dir = Path("preprocessed")
preprocessed_dir.mkdir(exist_ok=True)

community_means_csv = preprocessed_dir / "multitask_community_means.csv"
community_sizes_csv = preprocessed_dir / "multitask_community_sizes.csv"
design_matrices_npz = preprocessed_dir / "multitask_design_matrices.npz"
metadata_json = preprocessed_dir / "multitask_preprocessing_metadata.json"
summary_csv = preprocessed_dir / "multitask_preprocessing_summary.csv"
missing_values_csv = preprocessed_dir / "multitask_missing_values_before_imputation.csv"

task_names = ["SalePrice", "GrLivArea", "GarageArea", "TotalBsmtSF"]
n_communities = 20


## Clean and Cluster Ames Housing


In [ ]:
raw = pd.read_csv(data_csv)

column_map = {
    "SalePrice": "SalePrice",
    "Overall Qual": "OverallQual",
    "Gr Liv Area": "GrLivArea",
    "Garage Cars": "GarageCars",
    "Total Bsmt SF": "TotalBsmtSF",
    "Year Built": "YearBuilt",
    "TotRms AbvGrd": "TotRmsAbvGrd",
    "Bedroom AbvGr": "BedroomAbvGr",
    "Full Bath": "FullBath",
    "1st Flr SF": "FirstFlrSF",
    "2nd Flr SF": "SecondFlrSF",
    "Garage Area": "GarageArea",
    "Lot Area": "LotArea",
    "Fireplaces": "Fireplaces",
    "Overall Cond": "OverallCond",
}

selected = raw[list(column_map)].rename(columns=column_map).copy()
selected = selected.apply(pd.to_numeric, errors="coerce")

missing_before = selected.isna().sum().sort_values(ascending=False)
selected = selected.fillna(selected.median(numeric_only=True))
selected = selected.dropna().reset_index(drop=True)
full_clean_n = len(selected)


def kmeans_plus_plus_init(data, n_clusters, rng):
    centers = np.empty((n_clusters, data.shape[1]), dtype=float)
    centers[0] = data[rng.integers(data.shape[0])]
    closest_dist_sq = np.sum((data - centers[0]) ** 2, axis=1)

    for idx in range(1, n_clusters):
        probs = closest_dist_sq / closest_dist_sq.sum()
        centers[idx] = data[rng.choice(data.shape[0], p=probs)]
        dist_sq = np.sum((data - centers[idx]) ** 2, axis=1)
        closest_dist_sq = np.minimum(closest_dist_sq, dist_sq)

    return centers


def assign_clusters(data, centers):
    distances = np.sum((data[:, None, :] - centers[None, :, :]) ** 2, axis=2)
    return np.argmin(distances, axis=1)


def recompute_centers(data, labels, n_clusters, rng):
    centers = np.empty((n_clusters, data.shape[1]), dtype=float)
    for idx in range(n_clusters):
        members = data[labels == idx]
        if len(members) == 0:
            centers[idx] = data[rng.integers(data.shape[0])]
        else:
            centers[idx] = members.mean(axis=0)
    return centers


def run_kmeans(data, n_clusters, rng, max_iters=50, n_restarts=4):
    best_labels = None
    best_centers = None
    best_inertia = np.inf

    for _ in range(n_restarts):
        centers = kmeans_plus_plus_init(data, n_clusters, rng)
        labels = None
        for _ in range(max_iters):
            new_labels = assign_clusters(data, centers)
            new_centers = recompute_centers(data, new_labels, n_clusters, rng)
            if labels is not None and np.array_equal(new_labels, labels):
                centers = new_centers
                labels = new_labels
                break
            centers = new_centers
            labels = new_labels

        inertia = float(np.sum((data - centers[labels]) ** 2))
        if inertia < best_inertia:
            best_inertia = inertia
            best_labels = labels.copy()
            best_centers = centers.copy()

    return best_labels, best_centers, best_inertia


clustering_frame = selected.copy()
clustering_scale = clustering_frame.std(axis=0, ddof=0).replace(0, 1.0)
clustering_data = ((clustering_frame - clustering_frame.mean(axis=0)) / clustering_scale).to_numpy(dtype=float)
cluster_labels, cluster_centers, cluster_inertia = run_kmeans(clustering_data, n_communities, rng)

selected["cluster_id"] = cluster_labels
cluster_sizes = selected.groupby("cluster_id").size().rename("cluster_size")
community_means = selected.groupby("cluster_id", sort=True).mean(numeric_only=True)
community_means["cluster_size"] = cluster_sizes

community_means_with_size = community_means.reset_index(drop=True)
community_sizes = community_means_with_size.pop("cluster_size").to_numpy(dtype=int)
community_means_only = community_means_with_size.copy()

d = len(community_means_only)

print(f"Rows after preprocessing before clustering: {full_clean_n}")
print(f"Communities after k-means clustering: {d}")
print(f"Cluster size range: {community_sizes.min()} to {community_sizes.max()}")
print(f"K-means inertia on standardized features: {cluster_inertia:.3f}")
print("Missing values before imputation on selected columns:")
print(missing_before[missing_before > 0])
community_means_only.head()


## Build Task Design Matrices


In [ ]:
task_features = {
    "SalePrice": ["OverallQual", "GrLivArea", "GarageCars", "TotalBsmtSF", "YearBuilt"],
    "GrLivArea": ["TotRmsAbvGrd", "BedroomAbvGr", "FullBath", "FirstFlrSF", "SecondFlrSF"],
    "GarageArea": ["GarageCars", "OverallQual", "YearBuilt", "LotArea", "Fireplaces"],
    "TotalBsmtSF": ["LotArea", "YearBuilt", "FirstFlrSF", "OverallQual", "OverallCond"],
}

X_list = []
for task in task_names:
    X = community_means_only[task_features[task]].to_numpy(dtype=float)
    feature_std = X.std(axis=0)
    feature_std = np.where(feature_std == 0, 1.0, feature_std)
    X = (X - X.mean(axis=0)) / feature_std
    X = np.column_stack([np.ones(d), X])
    X_list.append(X)

design_summary = pd.DataFrame(
    {
        "task": task_names,
        "n_design_points": [X.shape[0] for X in X_list],
        "n_features_with_intercept": [X.shape[1] for X in X_list],
    }
)
design_summary


## Save Preprocessed Artifacts


In [ ]:
community_means_only.to_csv(community_means_csv, index=False)
pd.DataFrame({"cluster_id": np.arange(d), "cluster_size": community_sizes}).to_csv(
    community_sizes_csv, index=False
)
missing_before.rename("missing_before_imputation").to_frame().to_csv(missing_values_csv)
design_summary.to_csv(summary_csv, index=False)

np.savez(
    design_matrices_npz,
    task_names=np.array(task_names, dtype=object),
    community_sizes=community_sizes,
    **{f"X_{task}": X for task, X in zip(task_names, X_list)},
)

metadata = {
    "seed": SEED,
    "source_csv": str(data_csv),
    "n_rows_after_cleaning_before_clustering": int(full_clean_n),
    "n_communities": int(n_communities),
    "n_design_points": int(d),
    "cluster_size_min": int(community_sizes.min()),
    "cluster_size_max": int(community_sizes.max()),
    "cluster_inertia_standardized": float(cluster_inertia),
    "task_names": task_names,
    "task_features": task_features,
    "artifacts": {
        "community_means_csv": str(community_means_csv),
        "community_sizes_csv": str(community_sizes_csv),
        "design_matrices_npz": str(design_matrices_npz),
        "summary_csv": str(summary_csv),
        "missing_values_csv": str(missing_values_csv),
    },
}
metadata_json.write_text(json.dumps(metadata, indent=2), encoding="utf-8")

pd.DataFrame(
    {
        "artifact": [
            "community_means_csv",
            "community_sizes_csv",
            "design_matrices_npz",
            "metadata_json",
            "summary_csv",
            "missing_values_csv",
        ],
        "path": [
            community_means_csv,
            community_sizes_csv,
            design_matrices_npz,
            metadata_json,
            summary_csv,
            missing_values_csv,
        ],
    }
)
